# PyTorch Neural Network para Classificação Multiclasse

MLP com `torch.nn` para classificar entre **N classes** mutuamente exclusivas.

### Componentes principais
| Componente | Escolha | Por quê |
|---|---|---|
| Última camada | `nn.Linear(..., N_classes)` — **sem** Softmax | `CrossEntropyLoss` aplica internamente LogSoftmax |
| Loss | `nn.CrossEntropyLoss` | Combina `LogSoftmax` + `NLLLoss` — numericamente estável |
| Target dtype | `torch.LongTensor` | `CrossEntropyLoss` exige índices inteiros, não one-hot |
| Otimizador | `Adam` | Adaptativo, robusto |

### Por que NÃO usar Softmax explícito?
`CrossEntropyLoss` já inclui `LogSoftmax` internamente — usar Softmax antes causaria
dupla aplicação e instabilidade numérica.

## 1. Importar Bibliotecas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score,
                              confusion_matrix, classification_report)
from sklearn.datasets import make_classification

# Reprodutibilidade
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch versão: {torch.__version__}")

## 2. Gerar e Preparar Dados

Geramos um problema com **4 classes** e 10 features.

> **Atenção ao dtype:** targets devem ser `torch.LongTensor` (inteiros 64-bit) — requisito
> de `nn.CrossEntropyLoss`, que espera índices de classe, não vetores one-hot.

Isso difere da classificação binária, onde `y_train` era `FloatTensor` (para `BCELoss`).

In [ ]:
# 4 classes, 10 features (6 informativas)
X, y = make_classification(
    n_samples=500, n_features=10, n_informative=6,
    n_redundant=2, n_classes=4, n_clusters_per_class=1,
    random_state=42
)

num_classes = len(np.unique(y))
print(f"Classes únicas: {np.unique(y)}  →  {num_classes} classes")
print(f"Distribuição  : {np.bincount(y)}")

# Divisão treino / teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify: mantém proporção de classes
)

# Normalização
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Converter para tensores
X_train = torch.FloatTensor(X_train)
y_train = torch.LongTensor(y_train)    # LongTensor: índices inteiros para CrossEntropyLoss
X_test  = torch.FloatTensor(X_test)
y_test  = torch.LongTensor(y_test)

print(f"\nTreino : X={X_train.shape}, y={y_train.shape}  dtype={y_train.dtype}")
print(f"Teste  : X={X_test.shape},  y={y_test.shape}  dtype={y_test.dtype}")

## 3. Arquitetura da Rede Neural

**Diferença crítica** em relação à classificação binária:

- A saída tem **N neurônios** (um por classe) e produz **logits** (pontuações brutas)
- **NÃO** aplicamos Softmax aqui — `CrossEntropyLoss` faz isso internamente de forma
  numericamente estável

```
x₁ = ReLU(W₀·x  + b₀)
x₂ = ReLU(W₁·x₁ + b₁)
ŷ  =       W₂·x₂ + b₂    ← logits brutos, shape [N, num_classes]
```

A predição final é: `classe = argmax(ŷ)` — classe com maior logit.

In [ ]:
class MulticlassNN(nn.Module):
    """
    MLP para Classificação Multiclasse
    Saída: logits brutos [N, num_classes] — sem Softmax (CrossEntropyLoss inclui internamente)
    """
    def __init__(self, input_size, num_classes, hidden_size=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),                             # não-linearidade interna
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)             # logits: 1 neurônio por classe, sem ativação
        )

    def forward(self, x):
        return self.network(x)                     # retorna logits — CrossEntropyLoss aplica Softmax

model = MulticlassNN(input_size=10, num_classes=num_classes, hidden_size=64)
print(model)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal de parâmetros treináveis: {n_params:,}")
print(f"Saída do modelo para 1 amostra: shape {model(X_train[:1]).shape}  (logits por classe)")

## 4. Loss e Otimizador

**`nn.CrossEntropyLoss`** = `LogSoftmax` + `NLLLoss` combinados:
- Converte logits em log-probabilidades via `LogSoftmax`
- Calcula a *negative log-likelihood* da classe correta

**Por que usar `CrossEntropyLoss` e não `Softmax` + `NLLLoss` separados?**  
A implementação combinada é numericamente mais estável e mais eficiente.

In [ ]:
criterion = nn.CrossEntropyLoss()   # LogSoftmax + NLLLoss
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Loss function : {criterion}")
print(f"Otimizador    : Adam  (lr=0.001)")

## 5. Ciclo de Treino com Acompanhamento de Validação

Padrão padrão com a diferença que a **acurácia**
é calculada via `torch.max(outputs, dim=1)` — índice do logit máximo = classe predita.

In [ ]:
epochs       = 200
train_losses = []
val_losses   = []
train_accs   = []
val_accs     = []

for epoch in range(epochs):

    # ── Fase de Treino ────────────────────────────────────────────────
    model.train()
    outputs = model(X_train)                        # logits: shape [N, 4]
    loss    = criterion(outputs, y_train)           # CrossEntropyLoss aceita logits + LongTensor

    optimizer.zero_grad()                           # zera gradientes
    loss.backward()                                 # autograd
    optimizer.step()

    # Acurácia de treino: argmax dos logits = classe predita
    _, preds_train = torch.max(outputs.detach(), dim=1)
    acc_train = (preds_train == y_train).float().mean().item()
    train_losses.append(loss.item())
    train_accs.append(acc_train)

    # ── Fase de Validação — sem gradientes ───────────────────────────
    model.eval()
    with torch.no_grad():
        val_out  = model(X_test)
        val_loss = criterion(val_out, y_test)
        _, preds_val = torch.max(val_out, dim=1)
        acc_val = (preds_val == y_test).float().mean().item()

    val_losses.append(val_loss.item())
    val_accs.append(acc_val)

    if (epoch + 1) % 40 == 0:
        print(f"Epoch [{epoch+1:3d}/{epochs}] | "
              f"Train Loss: {loss.item():.4f}  Acc: {acc_train:.4f} | "
              f"Val Loss: {val_loss.item():.4f}  Acc: {acc_val:.4f}")

print("\nTreinamento concluído!")

## 6. Avaliação e Visualização

Para multiclasse usamos métricas **weighted** (ponderadas pelo suporte de cada classe):
- **`classification_report`**: visão completa por classe
- **Matriz de Confusão**: onde o modelo erra
- **Curvas de Loss e Acurácia**: verificação de overfitting

In [ ]:
model.eval()
with torch.no_grad():
    logits_train = model(X_train)
    logits_test  = model(X_test)
    _, y_pred_train = torch.max(logits_train, dim=1)
    _, y_pred_test  = torch.max(logits_test,  dim=1)

y_pred_train = y_pred_train.numpy()
y_pred_test  = y_pred_test.numpy()

# Métricas globais
acc_tr  = accuracy_score(y_train.numpy(), y_pred_train)
acc_te  = accuracy_score(y_test.numpy(),  y_pred_test)
prec_te = precision_score(y_test.numpy(), y_pred_test, average='weighted', zero_division=0)
rec_te  = recall_score(y_test.numpy(),    y_pred_test, average='weighted')
f1_te   = f1_score(y_test.numpy(),        y_pred_test, average='weighted')

print("=== Métricas Globais ===")
print(f"Acurácia Treino  : {acc_tr:.4f}")
print(f"Acurácia Teste   : {acc_te:.4f}")
print(f"Precisão (w) Teste: {prec_te:.4f}")
print(f"Recall   (w) Teste: {rec_te:.4f}")
print(f"F1-Score (w) Teste: {f1_te:.4f}")

print("\n=== Relatório por Classe ===")
print(classification_report(y_test.numpy(), y_pred_test,
                             target_names=[f'Classe {i}' for i in range(num_classes)]))

In [ ]:
cm = confusion_matrix(y_test.numpy(), y_pred_test)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Curvas de Loss ─────────────────────────────────────────────────────
axes[0].plot(train_losses, label='Treino',    color='royalblue')
axes[0].plot(val_losses,   label='Validação', color='darkorange', linestyle='--')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss (CrossEntropy)')
axes[0].set_title('Curvas de Loss\n(Overfitting?)')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# ── Plot 2: Curvas de Acurácia ────────────────────────────────────────────────
axes[1].plot(train_accs, label='Treino',    color='royalblue')
axes[1].plot(val_accs,   label='Validação', color='darkorange', linestyle='--')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia')
axes[1].set_title('Curvas de Acurácia\nTreino vs Validação')
axes[1].legend()
axes[1].grid(True, alpha=0.4)
axes[1].set_ylim(0, 1.05)

# ── Plot 3: Matriz de Confusão ────────────────────────────────────────────────
im = axes[2].imshow(cm, cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=axes[2])
axes[2].set_title(f'Matriz de Confusão\nAcurácia Teste: {acc_te:.4f}')
axes[2].set_xlabel('Predição')
axes[2].set_ylabel('Real')
ticks = list(range(num_classes))
labels = [f'Cl. {i}' for i in ticks]
axes[2].set_xticks(ticks); axes[2].set_xticklabels(labels)
axes[2].set_yticks(ticks); axes[2].set_yticklabels(labels)
thresh = cm.max() / 2
for i in range(num_classes):
    for j in range(num_classes):
        axes[2].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > thresh else 'black', fontsize=11)

plt.tight_layout()
plt.savefig('multiclasse_resultados.png', dpi=120, bbox_inches='tight')
plt.show()